# 3. Managed Identities — secrets without the secrets

The biggest problem with the previous notebook? **That `client_secret`.** You have to:

- Store it somewhere (Key Vault, hopefully)
- Rotate it on schedule
- Keep it out of logs, git, env var dumps
- Give every environment (dev/staging/prod) a different one

**Managed identities** solve this. Azure creates a service principal for your resource and hands your code a token *on demand*, with no secret in your code or config.

> This notebook is mostly conceptual — managed identities only exist inside Azure (your app has to be running on App Service / Container App / VM / Function). The mock server simulates the moving pieces so you can *see* the shapes; when deployed to Azure the code becomes simpler, not more complex.

## System-assigned vs User-assigned

| | System-assigned | User-assigned |
|-|-----------------|----------------|
| Lifecycle | Born with the resource, dies with it | Standalone Azure resource |
| Sharing | 1 identity per resource | Attach to many resources |
| Use when | Single app, simple case | Shared identity across VMs / Container Apps, or you want to pre-grant permissions before deploying code |
| In Bicep | `identity: { type: 'SystemAssigned' }` | `identity: { type: 'UserAssigned', userAssignedIdentities: { '<id>': {} } }` |

**Rule of thumb**: default to user-assigned. You can grant the permissions and use them from CI before your app even exists, and you can attach the same identity to blue/green deployments.

## How your code gets a token inside Azure

Every Azure compute host exposes a metadata endpoint **IMDS** (Instance Metadata Service) at a fixed internal address:

- VMs / VMSS: `http://169.254.169.254/metadata/identity/oauth2/token`
- App Service / Functions / Container Apps: `$IDENTITY_ENDPOINT` env var

Your code calls this endpoint with a resource URI. The platform (not your code, not Entra) handles signing and returns a real Entra access token.

You almost never call IMDS directly — the Azure SDK does it for you via `ManagedIdentityCredential` or, more commonly, `DefaultAzureCredential`.

## The exact same code, three environments

The magic of `DefaultAzureCredential`: it tries multiple credential sources in order, returning the first that works. You ship one binary and it figures out which credential to use based on where it's running.

```python
from azure.identity import DefaultAzureCredential
cred = DefaultAzureCredential()
token = cred.get_token('api://api-b/.default').token
```

Credential chain (simplified):

1. **EnvironmentCredential** — `AZURE_CLIENT_ID` / `SECRET` / `TENANT_ID` env vars
2. **WorkloadIdentityCredential** — federated identity (AKS, GitHub Actions OIDC)
3. **ManagedIdentityCredential** — IMDS inside Azure compute
4. **AzureCliCredential** — whatever `az login` cached on your laptop
5. **VSCodeCredential** / **AzurePowerShellCredential** / ...

So:
- On your laptop after `az login` → uses AzureCliCredential
- In a Container App with a managed identity → uses ManagedIdentityCredential
- In CI/CD → EnvironmentCredential or WorkloadIdentityCredential

**Zero code changes between them.**

## Simulating managed identity locally

> ### 🔬 What is real here and what is not
>
> **Managed identity cannot be simulated on your laptop.** Its whole security property is
> that the *Azure fabric* — not your code — holds the credential and proves the identity of
> the compute instance to Entra. There is no local equivalent, and any tutorial that claims
> otherwise is showing you something else.
>
> So the next cell is **not** a managed identity. It is a `client_credentials` call with a
> hard-coded secret, exactly like notebook 2, and it exists only to show that **the token a
> managed identity yields has the same shape** — same `aud`, same `roles`, no user claims —
> so the downstream API code is byte-for-byte identical either way.
>
> | | Real managed identity in Azure | This cell |
> |-|--------------------------------|-----------|
> | Credential | Held and rotated by Azure; your code never sees it | `daemon-secret-value`, in plain sight |
> | Token source | IMDS / `$IDENTITY_ENDPOINT` on the host | Our mock Entra's `/token` |
> | Who proves the identity | The Azure platform | Whoever knows the secret |
> | Resulting access token | app-only, `roles`, no `upn` | app-only, `roles`, no `upn` ← the only part that matches |
>
> We also can't run `DefaultAzureCredential` itself against the mock: `azure-identity`
> refuses non-HTTPS authorities, by design. The env vars we set below are therefore
> **inert** — nothing in this notebook reads them. They are there so you can see the names
> `EnvironmentCredential` looks for.

In [ ]:
import os, httpx, json

# These are the env var names EnvironmentCredential looks for. Setting them here is
# purely illustrative -- nothing below reads them back through azure-identity, because
# azure-identity will not talk to our plain-HTTP mock. In real Azure you would write:
#
#     from azure.identity import DefaultAzureCredential
#     token = DefaultAzureCredential().get_token('api://api-b/.default').token
#
# ...and on a Container App with a managed identity attached, that call goes to IMDS and
# no secret exists anywhere. Below is the raw stand-in, so we can compare token shapes.
os.environ['AZURE_CLIENT_ID']     = 'daemon-client-id'
os.environ['AZURE_CLIENT_SECRET'] = 'daemon-secret-value'   # <- a real MI has NO secret
os.environ['AZURE_TENANT_ID']     = 'contoso'

r = httpx.post('http://localhost:9100/contoso/oauth2/v2.0/token', data={
    'grant_type': 'client_credentials',
    'client_id': os.environ['AZURE_CLIENT_ID'],
    'client_secret': os.environ['AZURE_CLIENT_SECRET'],
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
token = r.json()['access_token']
print('Got token (SIMULATED managed-identity output - real MI needs no secret).')

# The claim shape is the part that transfers to real Azure: app-only, role-carrying.
import base64
payload = json.loads(base64.urlsafe_b64decode(
    token.split('.')[1] + '=' * (-len(token.split('.')[1]) % 4)))
assert payload['aud'] == 'api://api-b', f"aud must target the resource, got {payload['aud']}"
assert payload['roles'] == ['Files.Read.All'], f"expected an app role, got {payload.get('roles')}"
assert 'upn' not in payload, 'a managed-identity token carries no user - there is no user'

r = httpx.get('http://localhost:8002/files', headers={'Authorization': f'Bearer {token}'})
assert r.status_code == 200, f'expected api-b to accept the app-only token, got {r.status_code}'
assert r.json()['mode'] == 'app-only', f"expected app-only mode, got {r.json()['mode']}"
print(json.dumps(r.json(), indent=2))

## What the deployment looks like in Bicep

```bicep
// 1. Create a user-assigned managed identity
resource uami 'Microsoft.ManagedIdentity/userAssignedIdentities@2023-01-31' = {
  name: 'my-worker-identity'
  location: location
}

// 2. Attach it to a Container App
resource app 'Microsoft.App/containerApps@2024-03-01' = {
  name: 'worker'
  location: location
  identity: {
    type: 'UserAssigned'
    userAssignedIdentities: { '${uami.id}': {} }
  }
  properties: { /* ... image, env, ... */ }
}

// 3. In the Entra app registration for api-b, grant this identity the
//    'Files.Read.All' app role. This is done via an azureADServicePrincipal
//    appRoleAssignment - typically by a post-deployment script or az CLI.
```

## Granting app roles to a managed identity (CLI)

```bash
# Get the object id of the managed identity
MI_PRINCIPAL_ID=$(az identity show -g my-rg -n my-worker-identity --query principalId -o tsv)

# Get the service principal + role id for api-b
API_B_SP=$(az ad sp list --filter "displayName eq 'api-b'" --query '[0].id' -o tsv)
ROLE_ID=$(az ad sp show --id $API_B_SP --query "appRoles[?value=='Files.Read.All'].id" -o tsv)

# Assign
az rest --method POST \
  --uri "https://graph.microsoft.com/v1.0/servicePrincipals/$MI_PRINCIPAL_ID/appRoleAssignments" \
  --body "{\"principalId\":\"$MI_PRINCIPAL_ID\",\"resourceId\":\"$API_B_SP\",\"appRoleId\":\"$ROLE_ID\"}"
```

## Debug checklist when it doesn't work in Azure

1. Is the identity actually attached? `az containerapp identity show -g <rg> -n <app>`
2. Does the identity have the role granted? See CLI snippet above.
3. Is your code asking for the right `/.default` scope?
4. Does `aud` in the issued token match what the API validates? Use the `/debug/decode/<token>` endpoint or [jwt.ms](https://jwt.ms).
5. Is the JWKS URI reachable from your app's network?

## Summary

- Managed identity = no secrets, Azure-managed SP + IMDS.
- Prefer **user-assigned** for anything non-trivial.
- `DefaultAzureCredential` = one code path for local + Azure.
- In the token, a managed identity looks identical to any client-credentials token (`roles` claim, no user).